# Notebook 10: Consistency Distillation on MNIST

**目标**：用一个训好的 small DDPM 作 teacher，蒸馏出 1-step student。

**前置**：L13, derive_08

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import copy

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
ds = datasets.MNIST('./data', train=True, download=True, transform=tf)
loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2)

## 1. 简化 UNet（teacher 与 student 共用结构）

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        self.t_emb = nn.Sequential(nn.Linear(1, 64), nn.SiLU(), nn.Linear(64, ch*4))
        self.enc1 = nn.Conv2d(1, ch, 3, padding=1)
        self.enc2 = nn.Conv2d(ch, ch*2, 3, padding=1, stride=2)
        self.enc3 = nn.Conv2d(ch*2, ch*4, 3, padding=1, stride=2)
        self.mid = nn.Conv2d(ch*4, ch*4, 3, padding=1)
        self.dec3 = nn.ConvTranspose2d(ch*4*2, ch*2, 4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(ch*2*2, ch, 4, stride=2, padding=1)
        self.dec1 = nn.Conv2d(ch*2, 1, 3, padding=1)
    def forward(self, x, t):
        # Pad 28→32 for clean downsampling
        x = F.pad(x, (2, 2, 2, 2))
        emb = self.t_emb(t.float().unsqueeze(-1) / 1000.0).unsqueeze(-1).unsqueeze(-1)
        h1 = F.silu(self.enc1(x))
        h2 = F.silu(self.enc2(h1))
        h3 = F.silu(self.enc3(h2))
        h = F.silu(self.mid(h3) + emb)
        d3 = F.silu(self.dec3(torch.cat([h, h3], dim=1)))
        d2 = F.silu(self.dec2(torch.cat([d3, h2], dim=1)))
        out = self.dec1(torch.cat([d2, h1], dim=1))
        return out[:, :, 2:-2, 2:-2]  # 32→28

## 2. 训 Teacher (small DDPM)

In [ ]:
T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)

teacher = TinyUNet().to(device)
opt = torch.optim.Adam(teacher.parameters(), lr=2e-4)

import time
teacher.train()
for ep in range(2):  # 教学用，2 epoch 足够看效果
    t0 = time.time(); losses = []
    for x, _ in loader:
        x = x.to(device)
        t = torch.randint(0, T, (x.shape[0],), device=device)
        eps = torch.randn_like(x)
        xt = ac[t].view(-1,1,1,1).sqrt() * x + (1-ac[t]).view(-1,1,1,1).sqrt() * eps
        loss = F.mse_loss(teacher(xt, t), eps)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f'teacher ep {ep+1}: loss={np.mean(losses):.4f}, time={time.time()-t0:.0f}s')

## 3. Consistency Distillation 训练

核心 idea：让 student(x_{t+1}) ≈ student_EMA(x_t)，其中 x_t 由 teacher 走一步 ODE 得到。

In [ ]:
# Student 网络 + EMA 副本
student = TinyUNet().to(device)
student.load_state_dict(teacher.state_dict())  # 用 teacher 初始化（常用 trick）
student_ema = copy.deepcopy(student)
for p in student_ema.parameters():
    p.requires_grad = False

opt = torch.optim.Adam(student.parameters(), lr=1e-4)

# Karras-style sigma schedule (从 EDM)
N = 18  # 训练时离散化步数
sigma_min, sigma_max, rho = 0.002, 80.0, 7.0
i_arr = torch.arange(N, device=device, dtype=torch.float32)
sigmas = (sigma_max**(1/rho) + i_arr/(N-1) * (sigma_min**(1/rho) - sigma_max**(1/rho)))**rho

# Preconditioning (EDM)
sigma_data = 0.5
def precond_f(model, x, sigma):
    c_skip = sigma_data**2 / (sigma**2 + sigma_data**2)
    c_out = sigma * sigma_data / (sigma**2 + sigma_data**2).sqrt()
    c_in = 1.0 / (sigma**2 + sigma_data**2).sqrt()
    c_noise = 0.25 * sigma.log()
    # 把 sigma 换成 timestep-like input (用 c_noise * 1000 当 t)
    t_pseudo = (c_noise * 1000).clamp(0, 999).long()
    F_out = model(c_in.view(-1,1,1,1) * x, t_pseudo)
    return c_skip.view(-1,1,1,1) * x + c_out.view(-1,1,1,1) * F_out

# CD training loop
student.train()
for ep in range(2):
    t0 = time.time(); losses = []
    for x, _ in loader:
        x = x.to(device)
        # 随机选 n in [0, N-2]
        n = torch.randint(0, N - 1, (x.shape[0],), device=device)
        sigma_n = sigmas[n]
        sigma_np1 = sigmas[n + 1]  # 注意：sigmas 是降序的，所以 sigma_n < sigma_np1
        
        # 加噪到 sigma_np1 水平
        eps_noise = torch.randn_like(x)
        x_high = x + sigma_np1.view(-1,1,1,1) * eps_noise
        
        # Teacher 走一步反向 ODE：从 sigma_np1 到 sigma_n
        with torch.no_grad():
            t_high = torch.full((x.shape[0],), int((1 - sigma_np1[0].item()/sigma_max) * 999), device=device)
            eps_pred_high = teacher(x_high, t_high)  # noise prediction
            # 把 eps prediction 转成 score 然后走 ODE
            score = -eps_pred_high / sigma_np1.view(-1,1,1,1)
            x_low = x_high + (sigma_n - sigma_np1).view(-1,1,1,1) * (-sigma_np1.view(-1,1,1,1) * score)
            # 这等价于 x_low = x_high - (sigma_np1 - sigma_n) * eps_pred_high (Euler step)
        
        # Student 在两个点的预测
        pred_high = precond_f(student, x_high, sigma_np1)
        with torch.no_grad():
            pred_low = precond_f(student_ema, x_low, sigma_n)
        
        loss = F.mse_loss(pred_high, pred_low)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())
        
        # Update EMA
        with torch.no_grad():
            for p, pe in zip(student.parameters(), student_ema.parameters()):
                pe.data.mul_(0.99).add_(p.data, alpha=0.01)
    print(f'CD ep {ep+1}: loss={np.mean(losses):.4f}, time={time.time()-t0:.0f}s')

## 4. 1-step 采样对比

In [ ]:
@torch.no_grad()
def student_1step(n=8):
    student.eval()
    x = torch.randn(n, 1, 28, 28, device=device) * sigma_max
    return precond_f(student, x, torch.full((n,), sigma_max, device=device))

@torch.no_grad()
def teacher_full(n=8, n_steps=50):
    teacher.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    step = T // n_steps
    for ti in reversed(range(0, T, step)):
        t = torch.full((n,), ti, device=device, dtype=torch.long)
        eps = teacher(x, t)
        x0 = (x - (1-ac[ti]).sqrt() * eps) / ac[ti].sqrt()
        x0 = x0.clamp(-1, 1)
        if ti - step >= 0:
            x = ac[ti-step].sqrt() * x0 + (1 - ac[ti-step]).sqrt() * eps
        else:
            x = x0
    return x

fig, axes = plt.subplots(2, 8, figsize=(8, 2))
torch.manual_seed(42)
s = student_1step(8)
for j in range(8):
    axes[0][j].imshow((s[j,0].cpu()/2+0.5).clamp(0,1), cmap='gray'); axes[0][j].axis('off')
torch.manual_seed(42)
t_out = teacher_full(8, 50)
for j in range(8):
    axes[1][j].imshow((t_out[j,0].cpu()/2+0.5).clamp(0,1), cmap='gray'); axes[1][j].axis('off')
axes[0][0].set_ylabel('Student\n1 step', rotation=0, ha='right', va='center')
axes[1][0].set_ylabel('Teacher\n50 step', rotation=0, ha='right', va='center')
plt.tight_layout(); plt.show()

## 观察

- Student 1 步生成质量 < Teacher 50 步（**期望**，CD 是 distillation，存在 quality gap）
- 但 student 速度快 50 倍
- MNIST 任务太简单，效果不戏剧化；在 CIFAR/LDM 上 CD 的速度收益巨大

## 思考题

1. 把 EMA decay 从 0.99 改成 0.9 / 0.999，观察训练稳定性
2. 实现 multi-step CM sampling（4 步），与 1 步对比
3. 把 L2 loss 换成 LPIPS（如果有 GPU），观察 visual 质量提升
4. 把 CD 换成 CT（不用 teacher，用同一 noise 加到两个 sigma 水平）